In [1]:
"""
🧠 ALZHEIMER'S DETECTION - SEQUENTIAL TWO-STAGE FORENSIC PIPELINE (FEW-SHOT)
Hardware: NVIDIA RTX 5090 (32GB VRAM)
Architecture: Sequential Loading (VLM → LLM) for Maximum Intelligence

Phase 1: Qwen2.5-VL-3B-Instruct (Visual Ground Truth Extraction)
Phase 2: Qwen2.5-7B-Instruct (Few-Shot Forensic Linguistic Analysis)
Binary Classification: Control vs ProbableAD
"""

import os
import pandas as pd
import torch
import gc
import json
import re
from PIL import Image
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import AutoModelForImageTextToText, AutoProcessor
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
from tqdm import tqdm
import time

# ============================================================================
# CONFIGURATION - Using paths from existing code
# ============================================================================
CSV_PATH = r"D:\Dementia-Thesis - Don't access without permission\2_classes.csv"
IMAGE_PATH = r"D:\Dementia-Thesis - Don't access without permission\Cookie-Theft-stimulus.png"
TEXT_DIR = r"D:\Dementia-Thesis - Don't access without permission\cha_par_only"
OUTPUT_CSV = "alzheimer_forensic_qwen_few_shot.csv"

# Model IDs
VLM_MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"

# ============================================================================
# PHASE 1: THE EYE - Visual Ground Truth Extraction
# ============================================================================
def extract_visual_ground_truth(image_path):
    """
    Phase 1: Load VLM, extract ground truth, then completely unload.
    Returns: String describing all objects and actions in the image.
    """
    print("\n" + "="*80)
    print("🔬 PHASE 1: VISUAL GROUND TRUTH EXTRACTION")
    print("="*80)
    print(f"Loading VLM: {VLM_MODEL}")
    
    # Load VLM using AutoModelForImageTextToText (same as working notebook)
    vlm_model = AutoModelForImageTextToText.from_pretrained(
        VLM_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True
    )
    vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL)
    
    print(f"✅ VLM Loaded on GPU")
    print(f"🔋 GPU Memory: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    
    # Load image
    image = Image.open(image_path)
    
    # Create visual analysis prompt
    visual_prompt = """Analyze this 'Cookie Theft' image with forensic precision.

List EVERY visible object, person, and action in strict categories:

1. PEOPLE & ACTIONS:
   - What is each person doing?
   - Body positions and gestures

2. DANGER CUES (CRITICAL):
   - Is the stool tipping?
   - Is water overflowing?
   - Any unsafe situations?

3. OBJECTS & LOCATIONS:
   - Kitchen items (plates, cups, utensils)
   - Furniture
   - Windows, doors, environmental details

Output a comprehensive, factual list. This is GROUND TRUTH for medical diagnosis."""

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": visual_prompt}
            ]
        }
    ]
    
    # Process using the same approach as the working notebook
    text_prompt = vlm_processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = vlm_processor(text=[text_prompt], images=[image], padding=True, return_tensors="pt")
    inputs = {k: v.to(vlm_model.device) for k, v in inputs.items()}
    
    input_length = inputs["input_ids"].shape[-1]
    
    # Generate
    print("🔍 Extracting visual ground truth...")
    with torch.no_grad():
        generated_ids = vlm_model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.1,  # Low temperature for factual extraction
            do_sample=True,
            pad_token_id=vlm_processor.tokenizer.eos_token_id
        )
    
    ground_truth = vlm_processor.decode(generated_ids[0][input_length:], skip_special_tokens=True)
    
    print(f"\n✅ Ground Truth Extracted ({len(ground_truth)} chars)")
    print(f"Preview: {ground_truth[:200]}...")
    
    # ========================================================================
    # CRITICAL: COMPLETE MODEL UNLOAD TO FREE VRAM
    # ========================================================================
    print("\n🗑️ UNLOADING VLM FROM GPU...")
    del vlm_model
    del vlm_processor
    del inputs
    del generated_ids
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.synchronize()  # Ensure all GPU operations complete
    
    print(f"✅ VLM Unloaded. GPU Memory Freed: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print("="*80)
    
    return ground_truth


def parse_ground_truth(ground_truth_text):
    """
    Parse VLM ground truth into structured lists for easier LLM comparison.
    Returns: dict with categorized cues
    """
    parsed = {
        'people_actions': [],
        'danger_cues': [],
        'objects_locations': [],
        'raw_text': ground_truth_text
    }
    
    # Split by common patterns
    lines = ground_truth_text.split('\n')
    current_category = None
    
    for line in lines:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        
        # Detect category headers - MORE FLEXIBLE
        line_lower = line.lower()
        
        # People/Actions category
        if any(keyword in line_lower for keyword in ['people', 'person', 'action', 'boy', 'girl', 'woman', 'mother', 'child']):
            if any(keyword in line_lower for keyword in ['people', 'action', '1.', '**1']):
                current_category = 'people_actions'
                continue
        
        # Danger cues category
        if any(keyword in line_lower for keyword in ['danger', 'cue', 'unsafe', 'hazard', 'risk', '2.', '**2']):
            current_category = 'danger_cues'
            continue
        
        # Objects/Locations category
        if any(keyword in line_lower for keyword in ['object', 'location', 'kitchen', 'item', '3.', '**3']):
            current_category = 'objects_locations'
            continue
        
        # Extract content from list items
        if current_category:
            # Remove various bullet/number formats
            if line.startswith(('-', '•', '*', '1', '2', '3', '4', '5', '6', '7', '8', '9', '0')):
                content = re.sub(r'^[-•*\d\.\)\]}\s]+', '', line).strip()
                if content and len(content) > 3:  # Ignore very short items
                    parsed[current_category].append(content)
    
    # Enhanced fallback: Extract specific entities from raw text if categories are empty
    text_lower = ground_truth_text.lower()
    
    # Fallback for people/actions
    if not parsed['people_actions']:
        people_keywords = ['boy', 'girl', 'woman', 'mother', 'child', 'person', 'reaching', 'washing', 'standing', 'climbing']
        for line in lines:
            if any(keyword in line.lower() for keyword in people_keywords):
                content = line.strip()
                if content and not content.endswith(':') and len(content) > 5:
                    parsed['people_actions'].append(content)
    
    # Fallback for danger cues
    if not parsed['danger_cues']:
        danger_keywords = ['tip', 'tipping', 'overflow', 'overflowing', 'water', 'sink', 'stool', 'fall', 'falling', 'spill', 'unsafe', 'about to']
        for line in lines:
            line_lower = line.lower()
            if any(keyword in line_lower for keyword in danger_keywords):
                content = line.strip()
                if content and not content.endswith(':') and len(content) > 5:
                    parsed['danger_cues'].append(content)
    
    # Fallback for objects
    if not parsed['objects_locations']:
        object_keywords = ['cookie', 'jar', 'dish', 'plate', 'cup', 'window', 'curtain', 'cabinet', 'counter', 'sink', 'faucet', 'kitchen']
        for line in lines:
            line_lower = line.lower()
            if any(keyword in line_lower for keyword in object_keywords):
                content = line.strip()
                if content and not content.endswith(':') and len(content) > 5:
                    parsed['objects_locations'].append(content)
    
    # Deduplicate lists
    parsed['people_actions'] = list(dict.fromkeys(parsed['people_actions']))[:10]  # Max 10 items
    parsed['danger_cues'] = list(dict.fromkeys(parsed['danger_cues']))[:10]
    parsed['objects_locations'] = list(dict.fromkeys(parsed['objects_locations']))[:15]
    
    return parsed


# ============================================================================
# PHASE 2: THE BRAIN - Forensic Linguistic Analysis (FEW-SHOT)
# ============================================================================
def load_forensic_brain():
    """
    Phase 2: Load the large LLM for forensic analysis.
    Returns: (model, tokenizer)
    """
    print("\n" + "="*80)
    print("🧠 PHASE 2: LOADING FORENSIC BRAIN (FEW-SHOT)")
    print("="*80)
    print(f"Loading LLM: {LLM_MODEL}")
    
    tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
    llm_model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL,
        torch_dtype=torch.float16,
        device_map="cuda:0"
    )
    
    print(f"✅ LLM Loaded on GPU")
    print(f"🔋 GPU Memory: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print("="*80)
    
    return llm_model, tokenizer


def forensic_analysis_few_shot(llm_model, tokenizer, parsed_ground_truth, patient_transcript, few_shot_examples_text):
    """
    Perform FEW-SHOT forensic linguistic analysis with REAL examples from CSV.
    Args:
        parsed_ground_truth: dict with categorized cues
        patient_transcript: patient's transcription
        few_shot_examples_text: Real patient examples from CSV
    Returns: JSON dict with diagnosis and metrics.
    """
    
    # Format the structured ground truth for the prompt
    gt_formatted = f"""VISUAL GROUND TRUTH (Structured):

1. PEOPLE & ACTIONS:
{chr(10).join(f'   - {item}' for item in parsed_ground_truth['people_actions']) if parsed_ground_truth['people_actions'] else '   (No specific people/actions detected)'}

2. DANGER CUES (CRITICAL):
{chr(10).join(f'   - {item}' for item in parsed_ground_truth['danger_cues']) if parsed_ground_truth['danger_cues'] else '   (No danger cues detected)'}

3. OBJECTS & LOCATIONS:
{chr(10).join(f'   - {item}' for item in parsed_ground_truth['objects_locations']) if parsed_ground_truth['objects_locations'] else '   (No specific objects detected)'}
"""
    
    # FEW-SHOT prompt with REAL examples from CSV
    system_prompt = f"""### SYSTEM ROLE
You are a senior Neurologist specializing in Alzheimer's Disease (AD) detection. Analyze the patient's speech with OBJECTIVITY and PRECISION.
Be ACCURATE - do not bias toward either Control or ProbableAD. Follow the evidence strictly and learn from the examples below.

### INPUT DATA
{gt_formatted}

### FEW-SHOT EXAMPLES - Learn from these REAL patient cases:

{few_shot_examples_text}

### NOW ANALYZE THIS NEW PATIENT:

PATIENT TRANSCRIPT: "{patient_transcript}"

### ANALYSIS PROTOCOL
Compare this patient to the examples above and the visual ground truth.

STEP 1: CONTENT MAPPING (Accuracy)
- Compare the transcript to the visual reality above.
- **CRITICAL:** Did they miss "Danger Cues" listed above (especially stool tipping, water overflowing)?
- **CRITICAL:** Did they misidentify objects (Semantic Paraphasia)?
- Check if they mentioned the people/actions and objects from the lists.

STEP 2: LINGUISTIC BIOMARKER DETECTION (Style)
- **Anomia:** Count empty words ("thing", "stuff", "it", "they").
- **Fluency:** Count hesitations ("um", "uh") and fragments.
- **Syntax:** Is speech "Telegraphic" (broken grammar) or "Complex"?

STEP 3: BINARY CLASSIFICATION
- **Control:** Patient mentions most key objects (especially danger cues from the list), has fluent speech, and complex sentences (like Control examples above).
- **ProbableAD:** Patient misses danger cues from the list OR uses frequent empty words, hesitations, or broken grammar (like ProbableAD examples above).

STEP 4: MMSE ESTIMATION
- Estimate the patient's MMSE score (0-30) based on the transcript and comparing to the examples above with similar patterns.

### FINAL OUTPUT FORMAT (Strict JSON)
{{
    "missed_danger_cues": ["List of danger cues from above that patient failed to mention"],
    "anomia_count": (Integer),
    "syntax_quality": "High/Medium/Low",
    "clinical_score": (0-10 scale, where <5 is Control, >=5 is ProbableAD),
    "diagnosis": "Control" or "ProbableAD",
    "predicted_mmse": (Integer 0-30),
    "reasoning": "One sentence explanation comparing to examples above."
}}

Output ONLY the JSON, nothing else."""

    messages = [
        {"role": "system", "content": "You are a medical AI assistant."},
        {"role": "user", "content": system_prompt}
    ]
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to("cuda:0")
    
    with torch.no_grad():
        generated_ids = llm_model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.5,
            do_sample=True,
            top_p=0.9,
            top_k=40
        )
    
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]
    
    # Parse JSON response
    try:
        # Extract JSON from response (in case there's extra text)
        json_match = re.search(r'\{.*\}', response, re.DOTALL)
        if json_match:
            result = json.loads(json_match.group())
            return result
        else:
            print(f"⚠️ Failed to parse JSON, raw response: {response[:200]}")
            return {
                "missed_danger_cues": [],
                "anomia_count": 0,
                "syntax_quality": "Unknown",
                "clinical_score": 0,
                "diagnosis": "Error",
                "predicted_mmse": None,
                "reasoning": "JSON parsing failed"
            }
    except Exception as e:
        print(f"⚠️ Error parsing response: {str(e)}")
        return {
            "missed_danger_cues": [],
            "anomia_count": 0,
            "syntax_quality": "Unknown",
            "clinical_score": 0,
            "diagnosis": "Error",
            "predicted_mmse": None,
            "reasoning": f"Error: {str(e)}"
        }


# ============================================================================
# MAIN PIPELINE
# ============================================================================
def main():
    """Execute the full sequential pipeline with FEW-SHOT learning."""
    
    # ========================================================================
    # PHASE 1: Extract Visual Ground Truth (then unload)
    # ========================================================================
    ground_truth_raw = extract_visual_ground_truth(IMAGE_PATH)
    
    # Parse ground truth into structured categories
    print("\n📋 Parsing ground truth into structured categories...")
    parsed_ground_truth = parse_ground_truth(ground_truth_raw)
    print(f"✅ Extracted {len(parsed_ground_truth['danger_cues'])} danger cues")
    print(f"✅ Extracted {len(parsed_ground_truth['people_actions'])} people/actions")
    print(f"✅ Extracted {len(parsed_ground_truth['objects_locations'])} objects/locations")
    
    # Print what was actually extracted
    print("\n" + "="*80)
    print("🔍 VLM EXTRACTED GROUND TRUTH DETAILS")
    print("="*80)
    
    print("\n1️⃣ PEOPLE & ACTIONS:")
    if parsed_ground_truth['people_actions']:
        for item in parsed_ground_truth['people_actions']:
            print(f"   - {item}")
    else:
        print("   ⚠️ NONE DETECTED")
    
    print("\n2️⃣ DANGER CUES:")
    if parsed_ground_truth['danger_cues']:
        for item in parsed_ground_truth['danger_cues']:
            print(f"   - {item}")
    else:
        print("   ⚠️ NONE DETECTED")
    
    print("\n3️⃣ OBJECTS & LOCATIONS:")
    if parsed_ground_truth['objects_locations']:
        for item in parsed_ground_truth['objects_locations']:
            print(f"   - {item}")
    else:
        print("   ⚠️ NONE DETECTED")
    
    print("\n" + "="*80)
    print("📄 RAW VLM OUTPUT (first 800 chars):")
    print("="*80)
    print(parsed_ground_truth['raw_text'][:800])
    if len(parsed_ground_truth['raw_text']) > 800:
        print(f"\n... (truncated, total {len(parsed_ground_truth['raw_text'])} chars)")
    print("="*80)
    
    if len(parsed_ground_truth['people_actions']) == 0:
        print("\n⚠️ WARNING: VLM did not detect any people/actions!")
        print("Possible reasons:")
        print("  1. VLM output format doesn't match parsing patterns")
        print("  2. VLM didn't use expected headers like 'PEOPLE & ACTIONS'")
        print("  3. VLM output structure is different than expected")
        print("Check RAW VLM OUTPUT above to see actual format.")
    print("="*80)
    
    # ========================================================================
    # SELECT FEW-SHOT EXAMPLES FROM CSV (6 diverse examples with unique MMSE)
    # ========================================================================
    print("\n" + "="*80)
    print("📚 SELECTING FEW-SHOT EXAMPLES FROM CSV")
    print("="*80)
    
    # Load CSV for few-shot example selection
    df = pd.read_csv(CSV_PATH)
    df_gt_valid = df[df['dx'].notna()].copy()
    
    # Helper function to load .cha file content by ID
    def load_cha_content(file_id, text_directory):
        """Load the content of a .cha file given an ID."""
        file_path = os.path.join(text_directory, f"{file_id}.cha")
        if os.path.exists(file_path):
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    return f.read().strip()
            except Exception as e:
                print(f"Warning: Error reading {file_path}: {e}")
        return None
    
    # Get examples from CSV with valid MMSE scores and .cha files
    examples_pool_candidates = df_gt_valid[df_gt_valid['mmse'].notna()].copy()
    
    examples_pool = []
    for idx, row in examples_pool_candidates.iterrows():
        content = load_cha_content(str(row['id']), TEXT_DIR)
        if content and len(content) > 20:
            examples_pool.append(row)
    
    examples_pool = pd.DataFrame(examples_pool) if examples_pool else pd.DataFrame()
    print(f"✅ Found {len(examples_pool)} examples with valid MMSE + .cha file")
    
    # Separate by diagnosis
    control_pool = examples_pool[examples_pool['dx'].str.lower() == 'control'].copy()
    probablead_pool = examples_pool[examples_pool['dx'].str.lower() == 'probablead'].copy()
    
    print(f"  - Control pool: {len(control_pool)} (MMSE: {control_pool['mmse'].min():.0f}-{control_pool['mmse'].max():.0f})")
    print(f"  - ProbableAD pool: {len(probablead_pool)} (MMSE: {probablead_pool['mmse'].min():.0f}-{probablead_pool['mmse'].max():.0f})")
    
    # Sort by MMSE for diverse selection
    control_pool = control_pool.sort_values('mmse').reset_index(drop=True)
    probablead_pool = probablead_pool.sort_values('mmse').reset_index(drop=True)
    
    # Global MMSE tracking
    global_used_mmse = set()
    
    def select_diverse_examples(pool, n=3, global_mmse_tracker=None):
        """Select n examples with diverse MMSE scores"""
        if len(pool) < n:
            return pool
        
        if global_mmse_tracker is None:
            global_mmse_tracker = set()
        
        examples = []
        chunk_size = len(pool) // n
        used_ids = set()
        
        for i in range(n):
            start_idx = i * chunk_size
            end_idx = start_idx + chunk_size if i < n-1 else len(pool)
            chunk = pool.iloc[start_idx:end_idx]
            
            chunk = chunk[~chunk['id'].isin(used_ids)]
            chunk = chunk[~chunk['mmse'].isin(global_mmse_tracker)]
            
            if len(chunk) > 0:
                selected = chunk.sample(n=1, random_state=42+i)
                examples.append(selected)
                used_ids.add(selected['id'].values[0])
                global_mmse_tracker.add(selected['mmse'].values[0])
            else:
                remaining = pool[~pool['id'].isin(used_ids) & ~pool['mmse'].isin(global_mmse_tracker)]
                if len(remaining) > 0:
                    selected = remaining.sample(n=1, random_state=42+i+10)
                    examples.append(selected)
                    used_ids.add(selected['id'].values[0])
                    global_mmse_tracker.add(selected['mmse'].values[0])
        
        return pd.concat(examples) if examples else pool.head(n)
    
    # Select 3 from each class with globally unique MMSE
    control_examples = select_diverse_examples(control_pool, n=3, global_mmse_tracker=global_used_mmse)
    probablead_examples = select_diverse_examples(probablead_pool, n=3, global_mmse_tracker=global_used_mmse)
    
    few_shot_examples = pd.concat([control_examples, probablead_examples]).sample(frac=1, random_state=42)
    
    print(f"\n✅ Selected {len(few_shot_examples)} few-shot examples:")
    for idx, row in few_shot_examples.iterrows():
        cha_content = load_cha_content(str(row['id']), TEXT_DIR)
        print(f"  - ID {row['id']}: {row['dx']}, MMSE={row['mmse']:.0f}, length={len(cha_content) if cha_content else 0} chars")
    
    # Verify uniqueness
    unique_mmse = len(few_shot_examples['mmse'].unique())
    print(f"\n✅ Unique MMSE scores: {unique_mmse}/{len(few_shot_examples)}")
    if unique_mmse == len(few_shot_examples):
        print("✅ All MMSE scores are globally unique!")
    
    # Create few-shot examples text for the prompt
    def create_few_shot_examples_text():
        """Build few-shot examples from selected .cha files"""
        examples_text = ""
        
        for i, (idx, row) in enumerate(few_shot_examples.iterrows(), 1):
            cha_content = load_cha_content(str(row['id']), TEXT_DIR)
            
            if not cha_content:
                continue
            
            # Format diagnosis label with severity
            if row['dx'].lower() == 'control':
                diagnosis_label = "Control (Healthy)"
            elif row['dx'].lower() == 'probablead':
                if row['mmse'] >= 24:
                    diagnosis_label = "ProbableAD (Mild)"
                elif row['mmse'] >= 18:
                    diagnosis_label = "ProbableAD (Moderate)"
                else:
                    diagnosis_label = "ProbableAD (Severe)"
            else:
                diagnosis_label = row['dx']
            
            examples_text += f"""
**EXAMPLE {i}: {diagnosis_label}**
PATIENT TRANSCRIPT: "{cha_content}"

GROUND TRUTH:
{{
    "diagnosis": "{'Control' if row['dx'].lower() == 'control' else 'ProbableAD'}",
    "predicted_mmse": {int(row['mmse'])},
    "reasoning": "Real patient data - ID {row['id']}"
}}

---
"""
        
        return examples_text
    
    FEW_SHOT_EXAMPLES = create_few_shot_examples_text()
    print(f"\n✅ Few-shot prompt prepared with {len(few_shot_examples)} real examples")
    print("="*80)
    
    # ========================================================================
    # PHASE 2: Load Forensic Brain (LLM)
    # ========================================================================
    llm_model, tokenizer = load_forensic_brain()
    
    # ========================================================================
    # Load patient data
    # ========================================================================
    print("\n" + "="*80)
    print("📂 LOADING PATIENT DATA")
    print("="*80)
    
    df = pd.read_csv(CSV_PATH)
    print(f"✅ Loaded {len(df)} samples from CSV")
    
    # Filter CSV to only include rows with valid dx (not null)
    valid_df = df[df['dx'].notna()].copy()
    print(f"✅ Found {len(valid_df)} samples with valid dx labels in CSV")
    
    # Get valid IDs from CSV (convert to string and strip whitespace)
    valid_ids = set(str(id).strip() for id in valid_df['id'].values)
    print(f"✅ Valid IDs in CSV: {len(valid_ids)}")
    
    # Get all .cha files from directory
    all_cha_files = sorted([f for f in os.listdir(TEXT_DIR) if f.endswith('.cha')])
    print(f"✅ Total .cha files in directory: {len(all_cha_files)}")
    
    # ONLY process .cha files whose filename (without .cha) matches an ID in CSV
    text_files = []
    for filename in all_cha_files:
        file_id = filename.replace('.cha', '').strip()
        if file_id in valid_ids:
            text_files.append(filename)
    
    print(f"✅ Matched {len(text_files)} .cha files with CSV IDs")
    print(f"📊 Processing {len(text_files)} files")
    
    # ========================================================================
    # Process each patient transcript
    # ========================================================================
    results = []
    
    print("\n" + "="*80)
    print("🔬 FORENSIC ANALYSIS - PROCESSING PATIENTS (FEW-SHOT)")
    print("="*80)
    
    # Track processing times
    processing_times = []
    
    # Use tqdm for progress bar with time estimates
    for idx, filename in enumerate(tqdm(text_files, desc="Processing files", unit="file")):
        file_start_time = time.time()
        
        # Read transcript
        file_path = os.path.join(TEXT_DIR, filename)
        with open(file_path, 'r', encoding='utf-8') as f:
            patient_transcript = f.read().strip()
        
        # Get file ID (remove .cha extension)
        file_id = filename.replace('.cha', '').strip()
        
        # Get matched row from valid_df
        matched_rows = valid_df[valid_df['id'].astype(str).str.strip() == file_id]
        
        if len(matched_rows) == 0:
            tqdm.write(f"⚠️ [{idx+1}/{len(text_files)}] No matching CSV entry for {file_id}, skipping")
            continue
        
        matched_row = matched_rows.iloc[0]
        
        # Get ground truth labels (ProbableAD or Control) - case insensitive
        dx_value = str(matched_row['dx']).strip()
        dx_value_lower = dx_value.lower()
        
        if dx_value_lower == 'control':
            actual_label = 'Control'
        elif dx_value_lower == 'probablead':
            actual_label = 'ProbableAD'
        else:
            tqdm.write(f"⚠️ [{idx+1}/{len(text_files)}] Unknown diagnosis: {dx_value}, skipping")
            continue
        
        # Get MMSE if available, otherwise None
        actual_mmse = float(matched_row['mmse']) if pd.notna(matched_row['mmse']) else None
        
        # Run forensic analysis with few-shot examples
        analysis_result = forensic_analysis_few_shot(llm_model, tokenizer, parsed_ground_truth, patient_transcript, FEW_SHOT_EXAMPLES)
        
        # Compile results - normalize predicted diagnosis to match case
        predicted_diag = str(analysis_result.get('diagnosis', 'Error')).strip()
        predicted_diag_lower = predicted_diag.lower()
        
        # Normalize to standard case
        if predicted_diag_lower == 'control':
            predicted_diag = 'Control'
        elif predicted_diag_lower == 'probablead':
            predicted_diag = 'ProbableAD'
        elif predicted_diag_lower in ['error', 'unknown']:
            predicted_diag = predicted_diag.capitalize()
        
        result_row = {
            'filename': filename,
            'file_id': file_id,
            'predicted_diagnosis': predicted_diag,
            'predicted_mmse': analysis_result.get('predicted_mmse', None),
            'actual_diagnosis': actual_label,
            'actual_mmse': actual_mmse,
            'clinical_score': analysis_result.get('clinical_score', 0),
            'missed_danger_cues': ', '.join(analysis_result.get('missed_danger_cues', [])),
            'anomia_count': analysis_result.get('anomia_count', 0),
            'syntax_quality': analysis_result.get('syntax_quality', 'Unknown'),
            'reasoning': analysis_result.get('reasoning', ''),
            'vlm_ground_truth_raw': parsed_ground_truth['raw_text'],  # Raw VLM output
            'vlm_danger_cues': ' | '.join(parsed_ground_truth['danger_cues']),  # Parsed danger cues list
            'vlm_people_actions': ' | '.join(parsed_ground_truth['people_actions']),  # Parsed people/actions
            'vlm_objects': ' | '.join(parsed_ground_truth['objects_locations']),  # Parsed objects
            'raw_json': json.dumps(analysis_result)
        }
        
        results.append(result_row)
        
        # Calculate processing time for this file
        file_time = time.time() - file_start_time
        processing_times.append(file_time)
        avg_time = np.mean(processing_times)
        remaining_files = len(text_files) - (idx + 1)
        est_remaining_time = avg_time * remaining_files
        
        # Print result with timing info
        predicted_mmse_display = f"{analysis_result.get('predicted_mmse', 'N/A')}"
        actual_mmse_display = f"{actual_mmse if actual_mmse is not None else 'N/A'}"
        tqdm.write(f"✅ [{idx+1}/{len(text_files)}] {filename}: {predicted_diag} (Actual: {actual_label}) | "
                   f"MMSE: {predicted_mmse_display} (Actual: {actual_mmse_display}) | "
                   f"Time: {file_time:.1f}s | Avg: {avg_time:.1f}s/file | "
                   f"ETA: {est_remaining_time/60:.1f}min")
    
    # Print final timing statistics
    print(f"\n{'='*80}")
    print(f"⏱️ PROCESSING TIME STATISTICS")
    print(f"{'='*80}")
    if processing_times:
        print(f"Total files processed: {len(processing_times)}")
        print(f"Total time: {sum(processing_times)/60:.2f} minutes ({sum(processing_times):.1f} seconds)")
        print(f"Average time per file: {np.mean(processing_times):.2f} seconds")
        print(f"Fastest file: {np.min(processing_times):.2f} seconds")
        print(f"Slowest file: {np.max(processing_times):.2f} seconds")
        print(f"Median time: {np.median(processing_times):.2f} seconds")
    print(f"{'='*80}")
    
    # ========================================================================
    # Save results
    # ========================================================================
    print("\n" + "="*80)
    print("💾 SAVING RESULTS")
    print("="*80)
    results_df = pd.DataFrame(results)
    results_df.to_csv(OUTPUT_CSV, index=False)
    print("="*80)
    print(f"✅ Results saved to: {OUTPUT_CSV}")
    print(f"✅ Total processed: {len(results)} patients")
    
    # Calculate accuracy (treating Error/Unknown as incorrect predictions)
    if len(results) > 0:
        correct = sum(1 for r in results 
                     if r['predicted_diagnosis'] == r['actual_diagnosis'] 
                     and r['predicted_diagnosis'] not in ['Error', 'Unknown'])
        total = len(results)
        accuracy = correct / total * 100
        
        # Count valid MMSE predictions for regression metrics (both predicted AND actual must be valid)
        valid_mmse = [r for r in results 
                     if r['predicted_mmse'] is not None 
                     and not pd.isna(r['predicted_mmse'])
                     and r['actual_mmse'] is not None
                     and not pd.isna(r['actual_mmse'])]
        
        # ====================================================================
        # Generate and save visualizations
        # ====================================================================
        print("\n📊 GENERATING VISUALIZATIONS...")
        # Filter out Error/Unknown for clean confusion matrix
        valid_results = [r for r in results if r['predicted_diagnosis'] not in ['Error', 'Unknown']]
        
        if len(valid_results) > 0:
            y_true = [r['actual_diagnosis'] for r in valid_results]
            y_pred = [r['predicted_diagnosis'] for r in valid_results]
            
            # Confusion Matrix
            plt.figure(figsize=(10, 8))
            cm = confusion_matrix(y_true, y_pred, labels=['Control', 'ProbableAD'])
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                       xticklabels=['Control', 'ProbableAD'],
                       yticklabels=['Control', 'ProbableAD'],
                       cbar_kws={'label': 'Count'})
            plt.title('Confusion Matrix - Alzheimer\'s Forensic Detection (Few-Shot)\n(Qwen Sequential Pipeline)', 
                     fontsize=14, fontweight='bold')
            plt.ylabel('Actual Diagnosis', fontsize=12)
            plt.xlabel('Predicted Diagnosis', fontsize=12)
            plt.tight_layout()
            plt.savefig('alzheimer_forensic_qwen_few_shot_confusion_matrix.png', dpi=300, bbox_inches='tight')
            plt.close()
            print("✅ Saved: alzheimer_forensic_qwen_few_shot_confusion_matrix.png")
            
            # Classification Report
            report = classification_report(y_true, y_pred, 
                                          labels=['Control', 'ProbableAD'],
                                          output_dict=True)
            
            # Metrics Bar Chart
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            
            # Precision, Recall, F1-Score
            metrics_data = {
                'Control': [report['Control']['precision'], 
                           report['Control']['recall'], 
                           report['Control']['f1-score']],
                'ProbableAD': [report['ProbableAD']['precision'], 
                              report['ProbableAD']['recall'], 
                              report['ProbableAD']['f1-score']]
            }
            x = np.arange(3)
            width = 0.35
            axes[0].bar(x - width/2, metrics_data['Control'], width, label='Control', color='#2ecc71')
            axes[0].bar(x + width/2, metrics_data['ProbableAD'], width, label='ProbableAD', color='#e74c3c')
            axes[0].set_ylabel('Score', fontsize=11)
            axes[0].set_title('Classification Metrics by Class (Few-Shot)', fontsize=12, fontweight='bold')
            axes[0].set_xticks(x)
            axes[0].set_xticklabels(['Precision', 'Recall', 'F1-Score'])
            axes[0].legend()
            axes[0].set_ylim([0, 1.1])
            axes[0].grid(axis='y', alpha=0.3)
            
            # Overall Accuracy
            overall_acc = correct / total
            axes[1].bar(['Overall Accuracy'], [overall_acc], color='#3498db', width=0.5)
            axes[1].set_ylabel('Accuracy', fontsize=11)
            axes[1].set_title('Overall Diagnostic Accuracy (Few-Shot)', fontsize=12, fontweight='bold')
            axes[1].set_ylim([0, 1.1])
            axes[1].grid(axis='y', alpha=0.3)
            axes[1].text(0, overall_acc + 0.05, f'{overall_acc*100:.2f}%', 
                        ha='center', fontsize=12, fontweight='bold')
            
            plt.tight_layout()
            plt.savefig('alzheimer_forensic_qwen_few_shot_metrics.png', dpi=300, bbox_inches='tight')
            plt.close()
            print("✅ Saved: alzheimer_forensic_qwen_few_shot_metrics.png")
        
        # MMSE Scatter Plot (if valid predictions exist)
        if valid_mmse and len(valid_mmse) > 0:
            plt.figure(figsize=(10, 8))
            actual_mmse_vals = [r['actual_mmse'] for r in valid_mmse]
            predicted_mmse_vals = [r['predicted_mmse'] for r in valid_mmse]
            
            plt.scatter(actual_mmse_vals, predicted_mmse_vals, alpha=0.6, s=100, edgecolors='black')
            plt.plot([0, 30], [0, 30], 'r--', linewidth=2, label='Perfect Prediction')
            plt.xlabel('Actual MMSE Score', fontsize=12)
            plt.ylabel('Predicted MMSE Score', fontsize=12)
            plt.title('MMSE Prediction Performance (Few-Shot)\n(Qwen Sequential Pipeline)', 
                     fontsize=14, fontweight='bold')
            plt.xlim([0, 31])
            plt.ylim([0, 31])
            plt.grid(alpha=0.3)
            plt.legend(fontsize=11)
            
            # Add MAE text
            mae_val = sum(abs(r['predicted_mmse'] - r['actual_mmse']) for r in valid_mmse) / len(valid_mmse)
            plt.text(2, 28, f'MAE: {mae_val:.2f}', fontsize=12, 
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
            
            plt.tight_layout()
            plt.savefig('alzheimer_forensic_qwen_few_shot_mmse_scatter.png', dpi=300, bbox_inches='tight')
            plt.close()
            print("✅ Saved: alzheimer_forensic_qwen_few_shot_mmse_scatter.png")
        
        print("\n📁 All visualizations saved to current directory")
        
        # Print metrics summary
        print(f"\n🎯 METRICS SUMMARY (FEW-SHOT)")
        print(f"{'='*80}")
        print(f"Total Samples: {total}")
        print(f"Correct Predictions: {correct}")
        print(f"Error/Unknown Predictions: {sum(1 for r in results if r['predicted_diagnosis'] in ['Error', 'Unknown'])}")
        print(f"Diagnostic Accuracy: {accuracy:.2f}% ({correct}/{total})")
        
        if valid_mmse:
            mae = sum(abs(r['predicted_mmse'] - r['actual_mmse']) for r in valid_mmse) / len(valid_mmse)
            print(f"MMSE MAE (on {len(valid_mmse)} valid predictions): {mae:.2f}")
        print(f"{'='*80}")
    
    # ========================================================================
    # Cleanup
    # ========================================================================
    print("\n🗑️ Final cleanup...")
    del llm_model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    
    print("\n" + "🔥"*40)
    print("FEW-SHOT PIPELINE COMPLETE!")
    print("🔥"*40)


if __name__ == "__main__":
    main()

c:\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



🔬 PHASE 1: VISUAL GROUND TRUTH EXTRACTION
Loading VLM: Qwen/Qwen2.5-VL-3B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.09s/it]
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


✅ VLM Loaded on GPU
🔋 GPU Memory: 6.99 GB
🔍 Extracting visual ground truth...

✅ Ground Truth Extracted (864 chars)
Preview: ### 1. PEOPLE & ACTIONS:
- **Person 1** (standing on stool): Reaching into a cookie jar.
- **Person 2**: Standing next to Person 1, watching.
- **Person 3**: Standing near the sink, holding a towel an...

🗑️ UNLOADING VLM FROM GPU...
✅ VLM Unloaded. GPU Memory Freed: 0.01 GB

📋 Parsing ground truth into structured categories...
✅ Extracted 1 danger cues
✅ Extracted 3 people/actions
✅ Extracted 13 objects/locations

🔍 VLM EXTRACTED GROUND TRUTH DETAILS

1️⃣ PEOPLE & ACTIONS:
   - - **Person 1** (standing on stool): Reaching into a cookie jar.
   - - **Person 2**: Standing next to Person 1, watching.
   - - **Person 3**: Standing near the sink, holding a towel and a plate.

2️⃣ DANGER CUES:
   - Water overflowing**: Water is spilling from the sink, creating a puddle on the floor.

3️⃣ OBJECTS & LOCATIONS:
   - Cookie jar
   - Plates
   - Cups
   - Utensils (e.g., s

Loading checkpoint shards: 100%|██████████| 4/4 [00:12<00:00,  3.16s/it]


...

✅ [496/498] 709-2.cha: Control (Actual: Control) | MMSE: 27 (Actual: N/A) | Time: 2.3s | Avg: 2.5s/file | ETA: 0.1min


Processing files: 100%|█████████▉| 497/498 [21:07<00:02,  2.36s/file]

✅ [497/498] 711-0.cha: Control (Actual: ProbableAD) | MMSE: 26 (Actual: 25.0) | Time: 2.4s | Avg: 2.5s/file | ETA: 0.0min


Processing files: 100%|██████████| 498/498 [21:10<00:00,  2.55s/file]


✅ [498/498] 714-0.cha: Control (Actual: ProbableAD) | MMSE: 27 (Actual: 18.0) | Time: 2.7s | Avg: 2.5s/file | ETA: 0.0min

⏱️ PROCESSING TIME STATISTICS
Total files processed: 498
Total time: 21.16 minutes (1269.4 seconds)
Average time per file: 2.55 seconds
Fastest file: 1.92 seconds
Slowest file: 4.44 seconds
Median time: 2.43 seconds

💾 SAVING RESULTS
✅ Results saved to: alzheimer_forensic_qwen_few_shot.csv
✅ Total processed: 498 patients

📊 GENERATING VISUALIZATIONS...
✅ Saved: alzheimer_forensic_qwen_few_shot_confusion_matrix.png
✅ Saved: alzheimer_forensic_qwen_few_shot_metrics.png
✅ Saved: alzheimer_forensic_qwen_few_shot_mmse_scatter.png

📁 All visualizations saved to current directory

🎯 METRICS SUMMARY (FEW-SHOT)
Total Samples: 498
Correct Predictions: 301
Error/Unknown Predictions: 0
Diagnostic Accuracy: 60.44% (301/498)
MMSE MAE (on 417 valid predictions): 4.91

🗑️ Final cleanup...

🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
FEW-SHOT PIPELINE COMPLETE!
🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥

In [2]:
"""
📊 METRICS ANALYSIS - Load and display comprehensive metrics from Few-Shot results
"""

import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)
from scipy.stats import pearsonr

# Load results CSV
OUTPUT_CSV = "alzheimer_forensic_qwen_few_shot.csv"
df = pd.read_csv(OUTPUT_CSV)

print("="*80)
print("🎯 CLASSIFICATION METRICS (Control vs ProbableAD) - Few-Shot")
print("="*80)

# Filter out Error/Unknown predictions
valid_df = df[~df['predicted_diagnosis'].isin(['Error', 'Unknown'])].copy()

if len(valid_df) > 0:
    y_true = valid_df['actual_diagnosis'].values
    y_pred = valid_df['predicted_diagnosis'].values
    
    # Overall metrics
    accuracy = accuracy_score(y_true, y_pred)
    
    # Per-class metrics
    precision_control = precision_score(y_true, y_pred, pos_label='Control', zero_division=0)
    recall_control = recall_score(y_true, y_pred, pos_label='Control', zero_division=0)
    f1_control = f1_score(y_true, y_pred, pos_label='Control', zero_division=0)
    
    precision_ad = precision_score(y_true, y_pred, pos_label='ProbableAD', zero_division=0)
    recall_ad = recall_score(y_true, y_pred, pos_label='ProbableAD', zero_division=0)
    f1_ad = f1_score(y_true, y_pred, pos_label='ProbableAD', zero_division=0)
    
    # Macro averages
    precision_macro = (precision_control + precision_ad) / 2
    recall_macro = (recall_control + recall_ad) / 2
    f1_macro = (f1_control + f1_ad) / 2
    
    # Support (count per class)
    support_control = sum(y_true == 'Control')
    support_ad = sum(y_true == 'ProbableAD')
    
    print(f"\n📈 OVERALL ACCURACY: {accuracy*100:.2f}%")
    print(f"   ({sum(y_true == y_pred)}/{len(y_true)} correct predictions)")
    
    print(f"\n{'='*80}")
    print(f"{'Metric':<20} {'Control':<15} {'ProbableAD':<15} {'Macro Avg':<15}")
    print(f"{'='*80}")
    print(f"{'Precision':<20} {precision_control:<15.4f} {precision_ad:<15.4f} {precision_macro:<15.4f}")
    print(f"{'Recall':<20} {recall_control:<15.4f} {recall_ad:<15.4f} {recall_macro:<15.4f}")
    print(f"{'F1-Score':<20} {f1_control:<15.4f} {f1_ad:<15.4f} {f1_macro:<15.4f}")
    print(f"{'Support':<20} {support_control:<15d} {support_ad:<15d} {support_control+support_ad:<15d}")
    print(f"{'='*80}")
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred, labels=['Control', 'ProbableAD'])
    print(f"\n📊 CONFUSION MATRIX:")
    print(f"{'='*50}")
    print(f"                 Predicted Control  Predicted ProbableAD")
    print(f"Actual Control        {cm[0][0]:<15d} {cm[0][1]:<15d}")
    print(f"Actual ProbableAD     {cm[1][0]:<15d} {cm[1][1]:<15d}")
    print(f"{'='*50}")
    
    # Error analysis
    errors = sum(y_true != y_pred)
    error_rate = errors / len(y_true) * 100
    print(f"\n❌ ERROR ANALYSIS:")
    print(f"   Total Errors: {errors}/{len(y_true)} ({error_rate:.2f}%)")
    print(f"   False Positives (Control → ProbableAD): {cm[0][1]}")
    print(f"   False Negatives (ProbableAD → Control): {cm[1][0]}")

else:
    print("⚠️ No valid predictions found!")

# ============================================================================
print("\n" + "="*80)
print("📉 REGRESSION METRICS (MMSE Score Prediction) - Few-Shot")
print("="*80)

# Filter for valid MMSE predictions
valid_mmse = df[
    df['predicted_mmse'].notna() & 
    df['actual_mmse'].notna()
].copy()

if len(valid_mmse) > 0:
    y_true_mmse = valid_mmse['actual_mmse'].values
    y_pred_mmse = valid_mmse['predicted_mmse'].values
    
    # Calculate regression metrics
    mae = mean_absolute_error(y_true_mmse, y_pred_mmse)
    mse = mean_squared_error(y_true_mmse, y_pred_mmse)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true_mmse, y_pred_mmse)
    
    # Pearson correlation
    correlation, p_value = pearsonr(y_true_mmse, y_pred_mmse)
    
    print(f"\n📊 REGRESSION PERFORMANCE:")
    print(f"{'='*60}")
    print(f"{'Metric':<30} {'Value':<20}")
    print(f"{'='*60}")
    print(f"{'Mean Absolute Error (MAE)':<30} {mae:<20.4f}")
    print(f"{'Root Mean Squared Error (RMSE)':<30} {rmse:<20.4f}")
    print(f"{'Mean Squared Error (MSE)':<30} {mse:<20.4f}")
    print(f"{'R² Score':<30} {r2:<20.4f}")
    print(f"{'Pearson Correlation':<30} {correlation:<20.4f}")
    print(f"{'P-value':<30} {p_value:<20.6f}")
    print(f"{'='*60}")
    
    print(f"\n📊 MMSE STATISTICS:")
    print(f"{'='*60}")
    print(f"{'Statistic':<30} {'Actual MMSE':<15} {'Predicted MMSE':<15}")
    print(f"{'='*60}")
    print(f"{'Sample Size':<30} {len(y_true_mmse):<15d} {len(y_pred_mmse):<15d}")
    print(f"{'Mean':<30} {np.mean(y_true_mmse):<15.2f} {np.mean(y_pred_mmse):<15.2f}")
    print(f"{'Median':<30} {np.median(y_true_mmse):<15.2f} {np.median(y_pred_mmse):<15.2f}")
    print(f"{'Std Dev':<30} {np.std(y_true_mmse):<15.2f} {np.std(y_pred_mmse):<15.2f}")
    print(f"{'Min':<30} {np.min(y_true_mmse):<15.2f} {np.min(y_pred_mmse):<15.2f}")
    print(f"{'Max':<30} {np.max(y_true_mmse):<15.2f} {np.max(y_pred_mmse):<15.2f}")
    print(f"{'Range':<30} {np.max(y_true_mmse)-np.min(y_true_mmse):<15.2f} {np.max(y_pred_mmse)-np.min(y_pred_mmse):<15.2f}")
    print(f"{'='*60}")
    
    # Error distribution
    errors_mmse = y_pred_mmse - y_true_mmse
    print(f"\n📊 PREDICTION ERROR DISTRIBUTION:")
    print(f"{'='*60}")
    print(f"Mean Error (Bias):        {np.mean(errors_mmse):>10.2f}")
    print(f"Median Error:             {np.median(errors_mmse):>10.2f}")
    print(f"Error Std Dev:            {np.std(errors_mmse):>10.2f}")
    print(f"Max Overestimation:       {np.max(errors_mmse):>10.2f}")
    print(f"Max Underestimation:      {np.min(errors_mmse):>10.2f}")
    print(f"{'='*60}")
    
    # Interpretation
    print(f"\n💡 INTERPRETATION:")
    if r2 >= 0.7:
        print(f"   ✅ EXCELLENT correlation (R² = {r2:.4f})")
    elif r2 >= 0.5:
        print(f"   ✅ GOOD correlation (R² = {r2:.4f})")
    elif r2 >= 0.3:
        print(f"   ⚠️ MODERATE correlation (R² = {r2:.4f})")
    else:
        print(f"   ❌ WEAK correlation (R² = {r2:.4f})")
    
    if mae < 3:
        print(f"   ✅ EXCELLENT prediction accuracy (MAE = {mae:.2f})")
    elif mae < 5:
        print(f"   ✅ GOOD prediction accuracy (MAE = {mae:.2f})")
    elif mae < 7:
        print(f"   ⚠️ MODERATE prediction accuracy (MAE = {mae:.2f})")
    else:
        print(f"   ❌ POOR prediction accuracy (MAE = {mae:.2f})")

else:
    print("⚠️ No valid MMSE predictions found!")

print("\n" + "="*80)
print("✅ METRICS ANALYSIS COMPLETE (Few-Shot)")
print("="*80)

🎯 CLASSIFICATION METRICS (Control vs ProbableAD) - Few-Shot

📈 OVERALL ACCURACY: 60.44%
   (301/498 correct predictions)

Metric               Control         ProbableAD      Macro Avg      
Precision            0.5572          0.8021          0.6796         
Recall               0.9218          0.3020          0.6119         
F1-Score             0.6946          0.4387          0.5667         
Support              243             255             498            

📊 CONFUSION MATRIX:
                 Predicted Control  Predicted ProbableAD
Actual Control        224             19             
Actual ProbableAD     178             77             

❌ ERROR ANALYSIS:
   Total Errors: 197/498 (39.56%)
   False Positives (Control → ProbableAD): 19
   False Negatives (ProbableAD → Control): 178

📉 REGRESSION METRICS (MMSE Score Prediction) - Few-Shot

📊 REGRESSION PERFORMANCE:
Metric                         Value               
Mean Absolute Error (MAE)      4.9137              
Root Mean Squ